# 📑 EOM - Restructure Mapping Report
 
## 🔗 Links
- **[BRD Restructure Mapping - TBD](tbd)**  
  Comprehensive documentation outlining the business requirements for this report.
 
- **[Story Ticket - TBD](tbd)**  
  Details about the story ticket, including tasks and progress.
  
- **[UAT Findings Ticket - TBD]()**  
  A summary of findings from UAT.
 
---

In [0]:
# %pip install openpyxl
# %restart_python

In [0]:
import os
import sys

dbutils.widgets.text("run_date", "")
dbutils.widgets.text("adhoc_run", '')

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = "/Workspace" + notebook_path.rsplit("/HoistInvestorReports/", 1)[0] 
    sys.path.insert(0, os.path.join(repo_root, "aws_utils"))
    sys.path.insert(0, os.path.join(repo_root, "01-utils"))
except Exception:
    sys.path.append("../../aws_utils")
    sys.path.append("../../01-utils")

from aws_utils import get_env_prefix, get_env_fullname
from helpers import load_configs, build_portfolio_filter

env = get_env_prefix()

In [0]:
from datetime import datetime
from pytz import timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

athens_tz = timezone('Europe/Athens')
adhoc_run = dbutils.widgets.get("adhoc_run")

if adhoc_run == '1':
    dbutils.widgets.text("run_date", '')
    run_date = dbutils.widgets.get("run_date")
else:
    now_date = datetime.now(athens_tz).strftime('%Y-%m-%d')
    now_date_str = datetime.now(athens_tz).strftime('%Y%m%d')
    run_date = spark.sql(f"SELECT CurrentMonthLastWorkingDate FROM `{env}-cdh`.reference.ref_calendar WHERE CalendarDate = last_day(add_months(to_date('{now_date}'), -1))").first()["CurrentMonthLastWorkingDate"]

now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')

In [0]:
from typing import Any
from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace
)

from typing import Optional
import delta  # type: ignore
from pyspark.sql import SparkSession
import pandas as pd

In [0]:
## Portfolios In Scope - All HOIST portfolios
PortfolioName = "('Cell', 'Orbit', 'Pearl Portfolio A', 'Mercury', 'Light')"    

DecomissionDate = "2026-06-04"

##### Views

In [0]:
reassignments = spark.sql(f"""                             
        select distinct re.LoanID
        from `{env}-reportinputfiles`.investor.hoist_reassignments as re
          """)
reassignments.createOrReplaceTempView("reassignments")

dexposure = spark.sql(f"""
         SELECT *,
         '{run_date}' AS RunDate
         FROM `{env}-cdh`.derivative.get_all_exposures_by_business_date('{run_date}')
         WHERE sourcesystem = 'profits'
         AND PortfolioName in {PortfolioName}
         """)
dexposure.createOrReplaceTempView("dexposure")                  

restructuring = spark.sql(f"""
        select r.*
        from ( select r.*, rank() over (partition by NewExposureKey, oldexposurekey order by BusinessDate desc, case when RestructuringType IS NULL THEN 0 ELSE 1 END DESC) as Rank
            from  `{env}-cdh`.primary.restructuring r
            WHERE r.BusinessDate <= '{run_date}'
            AND r.PortfolioName IN {PortfolioName}
            ) r 
        LEFT JOIN reassignments as re on re.LoanID = r.OldExposureKey
        LEFT JOIN reassignments as re2 on re2.LoanID = r.NewExposureKey
        where rank = 1 and re.LoanID IS NULL and re2.LoanID IS NULL
""")
restructuring.createOrReplaceTempView("restructuring")

product = spark.sql(f"""
      select * 
      from (              
            select *, rank() over (partition by ProductKey order by BusinessDate desc) AS Rank
            from  `{env}-cdh`.primary.product 
            WHERE SourceSystem = 'profits'
            and BusinessDate <= '{run_date}' )
        where Rank = 1
""")
product.createOrReplaceTempView("product")

##### Aggregations

In [0]:
Final = spark.sql(f"""
        select distinct
            cast('{run_date}' as date) as BusinessDate, 
            res.PortfolioName,
            de.CDI ,
            res.NewAgreementKey,
            res.NewExposureKey,
            res.OldExposureKey,
            res.WriteOffExposureKey,
            de.productkey,
            p.ProductDescription, 
            p.SubProduct,
            de.ExposureRestructureType
        from restructuring AS res
        INNER JOIN dexposure as de on de.ExposureKey = res.NewExposureKey
        left join product as p on de.productkey = p.productkey 
        where (de.JUSTIFICATIONCODE IS NULL OR de.JUSTIFICATIONCODE NOT IN (43055,43056))
       
        """)
Final.createOrReplaceTempView("Final")

##### Append Results

In [0]:
target_table_path_restructuremapping = f"`{env}-cdh-delivery`.investorreports.hoist_RestructureMapping"

spark.sql(f"""DELETE FROM {target_table_path_restructuremapping} WHERE BusinessDate = '{run_date}' AND PortfolioName IN {PortfolioName}""")

Final = Final.withColumn("IngestionTimestamp", lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")))

Final.write \
        .format("delta") \
        .partitionBy("BusinessDate") \
        .mode("append") \
        .saveAsTable(target_table_path_restructuremapping) # in order to write in delta_lake


spark.sql(f"REPAIR TABLE {target_table_path_restructuremapping} SYNC METADATA")